In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csgraph
from sklearn.manifold import SpectralEmbedding
from sklearn.preprocessing import normalize

import numpy as np
import torch
from PIL import Image
import os
import json

#importing the used models
from transformers import BlipProcessor, BlipModel

In [46]:
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipModel.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

coco_root = "./data/"
img_dir = os.path.join(coco_root, "images/")
ann_path = os.path.join(coco_root, "annotations", "captions_val2014.json")

with open(ann_path, "r") as f:
    coco = json.load(f)

image_id_to_filename = {img['id']: img['file_name'] for img in coco['images']}

existing_filenames = set(os.listdir(img_dir))

existing_image_ids = {img_id for img_id, fname in image_id_to_filename.items() if fname in existing_filenames}

image_caption_pairs = [
    {
        "image_path": os.path.join(img_dir, image_id_to_filename[ann["image_id"]]),
        "caption": ann["caption"]
    }
    for ann in coco['annotations']
    if ann["image_id"] in existing_image_ids
]

print(f"Number of image-caption pairs: {len(image_caption_pairs)}")

embeddings = []
for idx, item in enumerate(image_caption_pairs[:100]):
    try:
        image = Image.open(item["image_path"]).convert("RGB")

        #Image embedding
        inputs_img = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs_img)
        image_embedding = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        image_vector = image_embedding.cpu().numpy().squeeze()

        #Caption embedding
        inputs_text = processor(text=item["caption"], return_tensors="pt").to(device)
        with torch.no_grad():
            text_features = model.get_text_features(**inputs_text)
        text_embedding = text_features / text_features.norm(p=2, dim=-1, keepdim=True)
        text_vector = text_embedding.cpu().numpy().squeeze()

        embeddings.append({
            "image_embedding": image_vector,
            "text_embedding": text_vector,
            "caption": item["caption"],
            "image_path": item["image_path"]
        })

        if idx % 10 == 0:
            print(f"Processed {idx} image-caption pairs")

    except Exception as e:
        print(f"Error processing {item['image_path']}: {e}")
print(f"Processed {len(embeddings)} embeddings in total.")


`BlipModel` is going to be deprecated in future release, please use `BlipForConditionalGeneration`, `BlipForQuestionAnswering` or `BlipForImageTextRetrieval` depending on your usecase.
Some weights of BlipModel were not initialized from the model checkpoint at Salesforce/blip-image-captioning-base and are newly initialized: ['logit_scale', 'text_model.embeddings.LayerNorm.bias', 'text_model.embeddings.LayerNorm.weight', 'text_model.embeddings.position_embeddings.weight', 'text_model.embeddings.word_embeddings.weight', 'text_model.encoder.layer.0.attention.output.LayerNorm.bias', 'text_model.encoder.layer.0.attention.output.LayerNorm.weight', 'text_model.encoder.layer.0.attention.output.dense.bias', 'text_model.encoder.layer.0.attention.output.dense.weight', 'text_model.encoder.layer.0.attention.self.key.bias', 'text_model.encoder.layer.0.attention.self.key.weight', 'text_model.encoder.layer.0.attention.self.query.bias', 'text_model.encoder.layer.0.attention.self.query.weight', 'text_mo

Number of image-caption pairs: 395
Processed 0 image-caption pairs
Processed 10 image-caption pairs
Processed 20 image-caption pairs
Processed 30 image-caption pairs
Processed 40 image-caption pairs
Processed 50 image-caption pairs
Processed 60 image-caption pairs
Processed 70 image-caption pairs
Processed 80 image-caption pairs
Processed 90 image-caption pairs
Processed 100 embeddings in total.


In [52]:
image_embeddings = np.array([item["image_embedding"] for item in embeddings])
text_embeddings = np.array([item["text_embedding"] for item in embeddings])

In [61]:
def apply_spectral_embedding(image_embeddings, text_embeddings, n_components=128):

    image_embeddings = normalize(image_embeddings)
    text_embeddings = normalize(text_embeddings)
    
    B = cosine_similarity(text_embeddings, image_embeddings)
    #making the values non negative before building bipartite graph
    B = np.clip(B, 0, None)
    
    n_texts = text_embeddings.shape[0]
    n_images = image_embeddings.shape[0]
    
    zero_text = np.zeros((n_texts, n_texts))
    zero_image = np.zeros((n_images, n_images))
    
    A = np.block([
        [zero_text,     B],
        [B.T,  zero_image]
    ])
    #Nodes without direct strong connections get a weak connection (preventing this here)
    epsilon = 1e-3
    A += epsilon
    np.fill_diagonal(A, epsilon)
    
    #Laplacian and degree matrix are computed by scikit
    spectral = SpectralEmbedding(n_components=n_components, affinity='precomputed')
    embedded = spectral.fit_transform(A)
    

    spectral_text = normalize(embedded[:n_texts])
    spectral_image = normalize(embedded[n_texts:])
    
    return spectral_image, spectral_text

In [62]:
spectral_image_embeddings, spectral_text_embeddings = apply_spectral_embedding(
    image_embeddings, text_embeddings
)